In [ ]:
import torch
import torch.nn as nn
import numpy as np
from utils import LoadDataset, awgn, signal_preprocess_for_cnn, normalization
import sr_pytorch as sr
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split
from dlmodels import ClassificationNet, Clf_Model, Clf_Model_reverse
from tqdm.notebook import tqdm
from scipy import signal
from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
fig = plt.figure(layout='constrained', figsize=(20, 20))
subfigs = fig.subfigures(4, 5, wspace=0.00)

win_len = 256
crop_ratio = 0.3

pkt_range=range(0, 200)
tx_range = np.arange(30, 40, dtype=int)


data_dir = Path("./Dataset/mul_rec_gx")

data_path_list = sorted(data_dir.glob("*_test.h5"))


premodel = ClassificationNet().to(device) 
# model = Clf_Model_reverse(premodel, 10, 2).to(device)
model = Clf_Model(premodel, 10).to(device)
model.load_state_dict(torch.load('./saved_models/direct_pluto1b200mini1_800cl_1.pth'))
model.eval()

acc_set = []
LoadDatasetObj = LoadDataset()
for i, data_path in enumerate(data_path_list):
    index_i, index_j = i // 5, i % 5
    [IQ_data_lopy, Dev_label_lopy] = LoadDatasetObj.load_iq_samples(data_path, 
                                                            tx_range, pkt_range)

    IQ_data = np.concatenate([IQ_data_lopy])
    Dev_label_lopy = Dev_label_lopy - tx_range[0]
    Dev_label = np.concatenate([Dev_label_lopy])

    classes_num = len(np.unique(Dev_label))

    data_test = normalization(IQ_data)
    # print(data_test.shape)
    spec = sr.spectrogram(data_test, win_len, crop_ratio)

    spec = signal_preprocess_for_cnn(spec).to(device)
    BATCH_SIZE = 32
    outputs = []
    with torch.no_grad():
        for start in range(0, spec.shape[0], BATCH_SIZE):
            end = start + BATCH_SIZE
            batch = spec[start:end]
            out, _ = model(batch)
            outputs.append(out.cpu())
    clf_output = torch.cat(outputs, dim=0)

    label = torch.from_numpy(Dev_label)
    label = label.to(device).squeeze()

    # print the accuracy of the classifier
    acc = accuracy_score(label.cpu(), clf_output.cpu().argmax(dim=1))
    acc_set.append(acc)
    
    # plot the confusion matrix without colour bar
    classes = np.arange(0, classes_num) + 1
    cm = confusion_matrix(label.cpu(), clf_output.cpu().argmax(dim=1))
    ax = subfigs[index_i, index_j].subplots()
    ax.imshow(cm, cmap='Blues')

    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels(list(map(str, classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_yticklabels(list(map(str, classes)))
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')
    # show the percentage of samples in each class
    for m in range(cm.shape[0]):
        for n in range(cm.shape[1]):
            plt.text(n, m, '{:.0f}'.format(cm[m, n]), ha='center', va='center')

    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    

plt.show()
print(acc_set)